## Accessories - Dream 2000

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re
import time

In [2]:
## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

In [3]:
Accessories = []
seen_links = set()

In [4]:
SEARCH = "mobile+accessories"
BASE_URL = f"https://dream2000.com/en/search?q={SEARCH}&type=product"
MAX_PAGES = 100   # safety limit

driver = webdriver.Chrome(options=options)
driver.get(BASE_URL)

In [5]:
JS_EXTRACT = r"""const seen = new Set();
const out = [];
document.querySelectorAll('a[href*="/products/"]').forEach(a => {
  const href = a.href.split('?')[0];
  if (seen.has(href)) return;
  // climb to the smallest ancestor that looks like a full product card
  let card = a;
  for (let i = 0; i < 8 && card.parentElement; i++) {
    card = card.parentElement;
    const links = new Set([...card.querySelectorAll('a[href*="/products/"]')].map(x => x.href.split('?')[0]));
    if (links.size > 1) { card = card.firstElementChild === a ? a : card; break; }
    if (/(LE|EGP)/.test(card.innerText) && card.querySelector('img')) break;
  }
  seen.add(href);
  out.push({
    href: href,
    text: card.innerText || "",
    name: (a.getAttribute('aria-label') || a.getAttribute('title') || a.innerText || "").trim(),
    image: (card.querySelector('img') || {}).src || ""
  });
});
return out;"""

In [6]:
def parse_card(c):
    text = c["text"]
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    # name: link text, else the longest line that isn't price/stock/button text
    name = c["name"]
    if not name:
        cand = [l for l in lines if not re.search(r"LE|EGP|stock|basket|option|pickup|unit price", l, re.I)]
        name = max(cand, key=len) if cand else ""

    # price: first number next to LE / EGP (current price comes before the old one)
    m = re.search(r"(?:LE|EGP)\s*([\d,]+(?:\.\d+)?)|([\d,]+(?:\.\d+)?)\s*(?:LE|EGP)", text)
    price = (m.group(1) or m.group(2)) if m else "Product Not Available"

    s = re.search(r"out of stock|low in stock|in stock", text, re.I)
    stock = s.group(0).title() if s else "Not Available"

    return {"Name": name, "Link": c["href"], "Image": c["image"], "Price": price, "Stock": stock}

In [7]:
## -- Scraping --
for page in range(1, MAX_PAGES + 1):
    driver.get(f"{BASE_URL}&page={page}")

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href*="/products/"]'))
        )
    except TimeoutException:
        print(f"Page {page}: no products found. Stopping.")
        break

    time.sleep(3)  # let prices / lazy images load
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    cards = driver.execute_script(JS_EXTRACT)
    page_products = 0

    for c in cards:
        if c["href"] in seen_links:
            continue
        seen_links.add(c["href"])
        Accessories.append(parse_card(c))
        page_products += 1

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Accessories)}"
    )

    if page_products == 0:   # page repeated / past the last page
        print("Scraping finished. No more new products.")
        break

driver.quit()

Page 1 finished successfully. Products collected on this page: 10. Total products collected: 10
Page 2 finished successfully. Products collected on this page: 10. Total products collected: 20
Page 3 finished successfully. Products collected on this page: 10. Total products collected: 30
Page 4 finished successfully. Products collected on this page: 10. Total products collected: 40
Page 5 finished successfully. Products collected on this page: 10. Total products collected: 50
Page 6 finished successfully. Products collected on this page: 10. Total products collected: 60
Page 7 finished successfully. Products collected on this page: 10. Total products collected: 70
Page 8 finished successfully. Products collected on this page: 10. Total products collected: 80
Page 9 finished successfully. Products collected on this page: 10. Total products collected: 90
Page 10 finished successfully. Products collected on this page: 10. Total products collected: 100
Page 11 finished successfully. Product

In [8]:
df_Accessories = pd.DataFrame(Accessories)
df_Accessories.head()

,Name,Link,Image,Price,Stock
0,Hoco CA81,https://dream2000.com/en/products/hoco-ca81-ca...,https://dream2000.com/cdn/shop/files/84__1.jpg...,81,Not Available
1,DJI Osmo Mobile 7p,https://dream2000.com/en/products/dji-osmo-mob...,https://dream2000.com/cdn/shop/files/new_proje...,"7,000.00",Not Available
2,DJI Osmo Mobile 6,https://dream2000.com/en/products/osmo-selfie-...,https://dream2000.com/cdn/shop/files/osmo-self...,6,Not Available
3,Ugreen LP290,https://dream2000.com/en/products/ugreen-lp290...,https://dream2000.com/cdn/shop/files/2025-02-2...,290,Not Available
4,Hoco CA79,https://dream2000.com/en/products/hoco-ca79-ma...,https://dream2000.com/cdn/shop/files/81.jpg?v=...,79,Not Available


In [9]:
df_Accessories.to_csv("dream2000_mobile_accessories.csv", index=False, encoding="utf-8-sig")

In [10]:
df = pd.read_csv("dream2000_mobile_accessories.csv", encoding="utf-8-sig")

In [11]:
df

,Name,Link,Image,Price,Stock
0,Hoco CA81,https://dream2000.com/en/products/hoco-ca81-ca...,https://dream2000.com/cdn/shop/files/84__1.jpg...,81,Not Available
1,DJI Osmo Mobile 7p,https://dream2000.com/en/products/dji-osmo-mob...,https://dream2000.com/cdn/shop/files/new_proje...,"7,000.00",Not Available
2,DJI Osmo Mobile 6,https://dream2000.com/en/products/osmo-selfie-...,https://dream2000.com/cdn/shop/files/osmo-self...,6,Not Available
3,Ugreen LP290,https://dream2000.com/en/products/ugreen-lp290...,https://dream2000.com/cdn/shop/files/2025-02-2...,290,Not Available
4,Hoco CA79,https://dream2000.com/en/products/hoco-ca79-ma...,https://dream2000.com/cdn/shop/files/81.jpg?v=...,79,Not Available
...,...,...,...,...,...
166,Onikuma Headphone K1 Gaming Wired Professional,https://dream2000.com/en/products/onikuma-head...,https://dream2000.com/cdn/shop/files/headphone...,470.00,Not Available
167,Onikuma X29 RGB Wired Gaming Headset with Nois...,https://dream2000.com/en/products/onikuma-x29-...,https://dream2000.com/cdn/shop/files/new_proje...,550.00,Not Available
168,"Amazfit Watch GTS 4 (1.75"")",https://dream2000.com/en/products/amazfit-watc...,https://dream2000.com/cdn/shop/files/amazfit-w...,"5,911.00",Low In Stock
169,Delonghi Oil Heater 9 Fins 2000 Watt TRRS0920,https://dream2000.com/en/products/delonghi-oil...,https://dream2000.com/cdn/shop/files/delonghi-...,0920,Low In Stock


In [1]:
import pandas as pd
df = pd.read_csv("dream2000_mobile_accessories.csv", encoding="utf-8-sig")
df.to_parquet("dream2000_mobile_accessories.parquet", index=False)